# PRO131 – Recepción de Servicios (v7, nov-2024)
Checklist/HMI piloto (on-site): Verificaciones → Gestión HES → Notificación para facturación → Derivación a PRO023.

> Nota: Este notebook modela el flujo operativo para ejecución y trazabilidad. Las descripciones y validaciones están basadas en PRO131 (PDF corporativo).


In [ ]:
import ipywidgets as widgets
from IPython.display import display
import datetime, json, uuid

EN_CURSO = "EN_CURSO"
BLOQUEADO = "BLOQUEADO"
DETENIDO_STOP = "DETENIDO_STOP"
FINALIZADO = "FINALIZADO"
DERIVADO_PRO023 = "DERIVADO_PRO023"

# -------------------------
# PRO131 – Recepción de Servicios (v7, nov-2024)
# Enfoque: operación on-site (Administrador de Contrato) + gates normativos + gestión HES (SAP)
# Transacciones (según PRO131):
# - F30/F30-1: verificaciones normativas (cuando aplique)
# - ML81N: generar Hoja de Entrada de Servicios (HES)
# - FIORI / ML81N / ML85: liberar HES (liberador DOA / dueño de CECO-PEP-Grafo)
# - ZFI_PANEL_CONTROL o ME23N: monitoreo de factura
# Salida: Deriva a PRO023 (Revisión y Registro de Facturas)
# -------------------------

# Convención nodos:
# - type = "task" (Acción + Validación)
# - type = "decision" (rombo): pregunta obligatoria + opciones que definen siguiente nodo
# - type = "end" (cierre / derivación)

NODOS = {'D0_proveedor_notifico_y_respaldos': {'type': 'decision',
                                       'titulo': 'Inicio – Verificar notificación y respaldos',
                                       'rol': 'Administrador de Contrato',
                                       'pregunta': '¿El proveedor notificó el término del servicio y entregó los '
                                                   'respaldos requeridos para revisión?',
                                       'opciones': [{'label': 'SÍ → Iniciar verificaciones del servicio',
                                                     'next': 'T1_verificar_servicio_vs_oc'},
                                                    {'label': 'NO → Informar al proveedor y bloquear avance',
                                                     'next': 'END_bloqueado_falta_respaldos'}],
                                       'ayuda': 'En enfoque on-site, esta condición es un gate: si no hay '
                                                'notificación/respaldos, no se puede iniciar verificación ni HES.'},
 'END_bloqueado_falta_respaldos': {'type': 'end',
                                   'titulo': 'Bloqueo: Falta notificación o respaldos',
                                   'rol': 'Administrador de Contrato',
                                   'mensaje': 'Informar al proveedor que debe notificar el término del servicio y '
                                              'entregar respaldos requeridos antes de continuar (dejar evidencia del '
                                              'requerimiento).',
                                   'estado_final': 'BLOQUEADO'},
 'T1_verificar_servicio_vs_oc': {'type': 'task',
                                 'titulo': 'Verificar servicio vs condiciones pactadas (OC)',
                                 'rol': 'Administrador de Contrato',
                                 'descripcion': 'Constatar que el servicio prestado se ajusta a las condiciones '
                                                'establecidas (alcance, plazos, calidad) según la Orden de Compra (OC) '
                                                'y/o contrato.',
                                 'acciones': ['Verificar en terreno (o evidencia equivalente) que el servicio fue '
                                              'ejecutado conforme a alcance, plazos y calidad pactados en la '
                                              'OC/contrato.',
                                              'Revisar respaldos técnicos/operacionales asociados al hito/avance del '
                                              'servicio (informes, protocolos, firmas u otros que apliquen).'],
                                 'validacion': '¿El servicio y sus respaldos técnicos/operacionales cumplen lo pactado '
                                               'en la OC/contrato?',
                                 'next': 'D1_aplica_ley_subcontratacion',
                                 'checklist': ['Verificar en terreno (o evidencia equivalente) que el servicio fue '
                                               'ejecutado conforme a alcance, plazos y calidad pactados en la '
                                               'OC/contrato.',
                                               'Revisar respaldos técnicos/operacionales asociados al hito/avance del '
                                               'servicio (informes, protocolos, firmas u otros que apliquen).']},
 'D1_aplica_ley_subcontratacion': {'type': 'decision',
                                   'titulo': 'Determinar verificación normativa aplicable',
                                   'rol': 'Administrador de Contrato',
                                   'pregunta': '¿El servicio está sujeto a la Ley de Subcontratación (verificación '
                                               'normativa RE.17)?',
                                   'opciones': [{'label': 'SÍ → Ejecutar verificación normativa RE.17 (F30/F30-1)',
                                                 'next': 'T2_verificacion_normativa_RE17'},
                                                {'label': 'NO → Ejecutar verificación normativa RE.16',
                                                 'next': 'T2b_verificacion_normativa_RE16'}],
                                   'ayuda': 'PRO131 define verificaciones normativas distintas según aplique o no Ley '
                                            'de Subcontratación.'},
 'T2_verificacion_normativa_RE17': {'type': 'task',
                                    'titulo': 'Verificación normativa RE.17 (Ley Subcontratación)',
                                    'rol': 'Administrador de Contrato',
                                    'descripcion': 'Validar cumplimiento normativo cuando aplica Ley de '
                                                   'Subcontratación, utilizando documentación/validaciones '
                                                   'correspondientes (F30/F30-1 u otros respaldos requeridos).',
                                    'acciones': ['Verificar documentación y/o certificados requeridos para '
                                                 'cumplimiento de obligaciones laborales y previsionales cuando '
                                                 'aplique (p.ej., F30/F30-1, según corresponda).',
                                                 'Asegurar que los respaldos exigidos estén vigentes y conformes al '
                                                 'requerimiento normativo.'],
                                    'validacion': '¿La verificación normativa RE.17 quedó conforme '
                                                  '(documentación/certificados vigentes y correctos)?',
                                    'next': 'D2_ambas_verificaciones_ok',
                                    'checklist': ['Verificar documentación y/o certificados requeridos para '
                                                  'cumplimiento de obligaciones laborales y previsionales cuando '
                                                  'aplique (p.ej., F30/F30-1, según corresponda).',
                                                  'Asegurar que los respaldos exigidos estén vigentes y conformes al '
                                                  'requerimiento normativo.']},
 'T2b_verificacion_normativa_RE16': {'type': 'task',
                                     'titulo': 'Verificación normativa RE.16 (No aplica Ley Subcontratación)',
                                     'rol': 'Administrador de Contrato',
                                     'descripcion': 'Validar cumplimiento normativo aplicable cuando NO corresponde '
                                                    'Ley de Subcontratación, según registro/verificación RE.16.',
                                     'acciones': ['Verificar requisitos normativos aplicables bajo RE.16 (según el '
                                                  'servicio).',
                                                  'Confirmar que la documentación requerida esté conforme.'],
                                     'validacion': '¿La verificación normativa RE.16 quedó conforme?',
                                     'next': 'D2_ambas_verificaciones_ok',
                                     'checklist': ['Verificar requisitos normativos aplicables bajo RE.16 (según el '
                                                   'servicio).',
                                                   'Confirmar que la documentación requerida esté conforme.']},
 'D2_ambas_verificaciones_ok': {'type': 'decision',
                                'titulo': 'Gate: Verificaciones completas',
                                'rol': 'Administrador de Contrato',
                                'pregunta': '¿Están correctas ambas verificaciones (técnica/contractual y normativa)?',
                                'opciones': [{'label': 'SÍ → Gestionar HES en SAP', 'next': 'T3_generar_HES_ML81N'},
                                             {'label': 'NO → Informar al proveedor para enmendar y revalidar',
                                              'next': 'T2c_informar_discrepancias_y_loop'}],
                                'ayuda': 'Según PRO131: si no existe V°B° para cualquiera de las verificaciones, se '
                                         'informa al proveedor para que enmiende la situación.'},
 'T2c_informar_discrepancias_y_loop': {'type': 'task',
                                       'titulo': 'Informar discrepancias al proveedor y solicitar corrección (loop)',
                                       'rol': 'Administrador de Contrato',
                                       'descripcion': 'Si existe incumplimiento técnico/contractual o normativo, se '
                                                      'informa formalmente al proveedor para que proceda a enmendar la '
                                                      'situación antes de continuar.',
                                       'acciones': ['Informar formalmente al proveedor las discrepancias '
                                                    '(técnicas/contractuales y/o normativas).',
                                                    'Solicitar corrección/entrega de respaldos complementarios según '
                                                    'corresponda.',
                                                    'Registrar evidencia de la notificación (correo, acta, ticket u '
                                                    'otro medio definido).'],
                                       'validacion': '¿El proveedor fue informado formalmente y quedó registrada '
                                                     'evidencia de la solicitud de corrección?',
                                       'next': 'T1_verificar_servicio_vs_oc',
                                       'checklist': ['Informar formalmente al proveedor las discrepancias '
                                                     '(técnicas/contractuales y/o normativas).',
                                                     'Solicitar corrección/entrega de respaldos complementarios según '
                                                     'corresponda.',
                                                     'Registrar evidencia de la notificación (correo, acta, ticket u '
                                                     'otro medio definido).']},
 'T3_generar_HES_ML81N': {'type': 'task',
                          'titulo': 'Generar Hoja de Entrada de Servicios (HES) – ML81N',
                          'rol': 'Administrador de Contrato',
                          'descripcion': 'Luego de obtener V°B° de verificaciones, generar HES en SAP mediante '
                                         'transacción ML81N. El valor debe ser según lo pactado (cotización, estado de '
                                         'pago, avance/proporcionalidad). Adjuntar respaldos requeridos (estados de '
                                         'pago, certificados, F30/F31 cuando aplique).',
                          'acciones': ['Ingresar a SAP y generar HES mediante transacción ML81N.',
                                       'Cargar cantidades/montos según lo pactado (cotización/estado de '
                                       'pago/avance/proporcionalidad del servicio).',
                                       'Adjuntar respaldos requeridos en el sistema (estados de pago, certificados y, '
                                       'cuando aplique, formularios/certificados como F30/F31).'],
                          'validacion': '¿La HES fue creada en ML81N con valores correctos y respaldos adjuntos según '
                                        'corresponda?',
                          'next': 'T4_evaluar_proveedor_en_SAP',
                          'checklist': ['Ingresar a SAP y generar HES mediante transacción ML81N.',
                                        'Cargar cantidades/montos según lo pactado (cotización/estado de '
                                        'pago/avance/proporcionalidad del servicio).',
                                        'Adjuntar respaldos requeridos en el sistema (estados de pago, certificados y, '
                                        'cuando aplique, formularios/certificados como F30/F31).'],
                          'inputs': [{'key': 'numero_hes',
                                      'label': 'Ingresa numero de HES',
                                      'required': True,
                                      'multiline': False}]},
 'T4_evaluar_proveedor_en_SAP': {'type': 'task',
                                 'titulo': 'Evaluar proveedor en SAP (obligatorio)',
                                 'rol': 'Administrador de Contrato',
                                 'descripcion': 'Registrar evaluación del proveedor en SAP ingresando criterios '
                                                'definidos en el Anexo 1. La evaluación es requisito para proceder a '
                                                'grabación y posterior liberación de la HES. El proveedor debe ser '
                                                'evaluado al menos una vez por contrato/servicio.',
                                 'acciones': ['Registrar evaluación del proveedor en SAP utilizando criterios '
                                              'definidos (ver Anexo 1 del PRO131).',
                                              'Asegurar que la evaluación corresponde al servicio/hito recepcionado y '
                                              'queda asociada al contrato/OC.'],
                                 'validacion': '¿La evaluación del proveedor quedó registrada en SAP según Anexo 1 y '
                                               'es consistente con el servicio recepcionado?',
                                 'next': 'T5_grabar_HES',
                                 'checklist': ['Registrar evaluación del proveedor en SAP utilizando criterios '
                                               'definidos (ver Anexo 1 del PRO131).',
                                               'Asegurar que la evaluación corresponde al servicio/hito recepcionado y '
                                               'queda asociada al contrato/OC.']},
 'T5_grabar_HES': {'type': 'task',
                   'titulo': 'Grabar HES en SAP',
                   'rol': 'Administrador de Contrato',
                   'descripcion': 'Luego de ingresar datos y evaluación, grabar/guardar la HES para generar número de '
                                  'documento y permitir el flujo de liberación.',
                   'acciones': ['Guardar/grabar la HES en SAP con todos los datos requeridos y evaluación registrada.',
                                'Verificar que se generó número de HES para seguimiento.'],
                   'validacion': '¿La HES quedó grabada en SAP y se generó el número de documento?',
                   'next': 'T6_liberar_HES_DOA',
                   'checklist': ['Guardar/grabar la HES en SAP con todos los datos requeridos y evaluación registrada.',
                                 'Verificar que se generó número de HES para seguimiento.']},
 'T6_liberar_HES_DOA': {'type': 'task',
                        'titulo': 'Liberar (aceptar) HES (FIORI / ML81N / ML85)',
                        'rol': 'Liberador (DOA / dueño CECO-PEP-Grafo)',
                        'descripcion': 'La liberación es realizada por el usuario responsable del objeto de costo '
                                       '(dueño CECO, PEP o Grafo). En ciertos casos (gasto anticipado/pasivos uso '
                                       'corriente), el liberador será el indicado por el Administrador de Contrato '
                                       'dentro de la lista de liberadores disponibles.',
                        'acciones': ['Gestionar liberación electrónica de la HES por el liberador correspondiente '
                                     '(según DOA / responsable del objeto de costo).',
                                     'Confirmar que la HES quedó liberada para formalizar recepción conforme y '
                                     'habilitar el devengo.'],
                        'validacion': '¿La HES quedó liberada por el liberador correspondiente (DOA) en el sistema?',
                        'next': 'T7_comunicar_evaluacion_al_proveedor',
                        'checklist': ['Gestionar liberación electrónica de la HES por el liberador correspondiente '
                                      '(según DOA / responsable del objeto de costo).',
                                      'Confirmar que la HES quedó liberada para formalizar recepción conforme y '
                                      'habilitar el devengo.']},
 'T7_comunicar_evaluacion_al_proveedor': {'type': 'task',
                                          'titulo': 'Comunicar evaluación al proveedor',
                                          'rol': 'Administrador de Contrato',
                                          'descripcion': 'Notificar formalmente al proveedor el resultado de la '
                                                         'evaluación de desempeño. Mantener evidencia de comunicación '
                                                         '(según nota del PRO131).',
                                          'acciones': ['Enviar comunicación formal al proveedor con resultado de '
                                                       'evaluación.',
                                                       'Guardar evidencia/registro de la comunicación '
                                                       '(adjunto/registro según práctica definida).'],
                                          'validacion': '¿La evaluación fue comunicada al proveedor y existe '
                                                        'evidencia/registro?',
                                          'next': 'T8_notificar_HES_liberada',
                                          'checklist': ['Enviar comunicación formal al proveedor con resultado de '
                                                        'evaluación.',
                                                        'Guardar evidencia/registro de la comunicación '
                                                        '(adjunto/registro según práctica definida).']},
 'T8_notificar_HES_liberada': {'type': 'task',
                               'titulo': 'Notificar HES liberada (número HES)',
                               'rol': 'Administrador de Contrato',
                               'descripcion': 'Notificar al proveedor la creación/liberación de la HES (número) '
                                              'mediante correo electrónico para habilitar emisión de factura.',
                               'acciones': ['Enviar correo al proveedor informando número de HES liberada y '
                                            'referencias necesarias (OC/contrato si aplica).'],
                               'validacion': '¿El proveedor fue notificado con el número de HES liberada?',
                               'next': 'T9_monitorear_factura',
                               'checklist': ['Enviar correo al proveedor informando número de HES liberada y '
                                             'referencias necesarias (OC/contrato si aplica).']},
 'T9_monitorear_factura': {'type': 'task',
                           'titulo': 'Monitorear emisión/disponibilidad de factura',
                           'rol': 'Administrador de Contrato',
                           'descripcion': 'Monitorear que el proveedor emita factura luego de la notificación de HES. '
                                          'Revisar disponibilidad en SAP mediante transacción ZFI_PANEL_CONTROL o '
                                          'ME23N, verificando consistencia entre factura, OC y HES.',
                           'acciones': ['Monitorear emisión de factura del proveedor posterior a la notificación de '
                                        'HES.',
                                        'Verificar en SAP (ZFI_PANEL_CONTROL o ME23N) si la factura/documento está '
                                        'disponible.',
                                        'Realizar chequeo de consistencia básica (monto y referencias) entre factura, '
                                        'OC y HES liberada.'],
                           'validacion': '¿La factura está disponible en SAP y es consistente con OC/HES (sin '
                                         'diferencias evidentes)?',
                           'next': 'END_derivar_PRO023',
                           'checklist': ['Monitorear emisión de factura del proveedor posterior a la notificación de '
                                         'HES.',
                                         'Verificar en SAP (ZFI_PANEL_CONTROL o ME23N) si la factura/documento está '
                                         'disponible.',
                                         'Realizar chequeo de consistencia básica (monto y referencias) entre factura, '
                                         'OC y HES liberada.']},
 'END_derivar_PRO023': {'type': 'end',
                        'titulo': 'Derivar a PRO023 – Revisión y Registro de Facturas',
                        'rol': 'Finanzas / Contabilidad',
                        'mensaje': 'Con la factura ya disponible en el sistema, continuar con PRO023 (Revisión y '
                                   'Registro de Facturas) para cruce final y programación de pago.',
                        'estado_final': 'DERIVADO_PRO023'}}

# -------------------------
# Motor HMI (base: igual a piloto)
# -------------------------

def _now_iso():
    return datetime.datetime.now().isoformat(timespec="seconds")

class PRO131HMI:
    def __init__(self):
        self.nodo_id = "D0_proveedor_notifico_y_respaldos"
        self.historial = []
        self.logs = []
        self.output = widgets.Output(layout={"width":"100%"})

        self.btn_si = widgets.Button(description="SÍ", button_style="success", layout={"width":"48%","height":"44px"})
        self.btn_no = widgets.Button(description="NO", button_style="danger", layout={"width":"48%","height":"44px"})
        self.btn_volver = widgets.Button(description="Volver al paso anterior", layout={"width":"100%","height":"40px"})
        self.btn_exportar = widgets.Button(description="Exportar JSON (trazabilidad)", icon="download", layout={"width":"100%","height":"40px"})

        self.msg_box = widgets.HTML("")
        self.main_box = widgets.VBox([])
        self._decision_widget = None
        self._check_widgets = []
        self._input_widgets = []

        self.is_blocked = False
        self.block_reason = None
        self.block_panel = widgets.VBox([])
        self.btn_rehacer = widgets.Button(description="Rehacer paso", button_style="info", layout={"width":"100%","height":"40px"})
        self.btn_rehacer.on_click(self._on_rehacer)

        self._wire()
        self._render()

    def _wire(self):
        self.btn_si.on_click(self._on_si)
        self.btn_no.on_click(self._on_no)
        self.btn_volver.on_click(self._on_volver)
        self.btn_exportar.on_click(self._on_exportar)

    def _log(self, tipo, data=None):
        self.logs.append({
            "ts": _now_iso(),
            "tipo": tipo,
            "nodo_id": self.nodo_id,
            "data": data or {}
        })

    def _push_hist(self, prev_id):
        self.historial.append(prev_id)

    def _pop_hist(self):
        if self.historial:
            return self.historial.pop()
        return None

    def _set_msg(self, html):
        self.msg_box.value = html

    def _clear_msg(self):
        self.msg_box.value = ""

    def _render_header(self, n):
        badge = f"<span style='display:inline-block;padding:4px 10px;border-radius:999px;background:#eef2ff;border:1px solid #c7d2fe;font-size:12px;'><b>ROL:</b> {n.get('rol','')}</span>"
        return widgets.HTML(f"""
        <div style="padding:16px;border-radius:12px;background:#f8fafc;border:1px solid #e2e8f0;">
            <div style="font-size:12px;color:#0f172a;"><b>PRO131</b> – Recepción de Servicios (v7, nov-2024)</div>
            <div style="margin-top:6px;font-size:22px;color:#0f172a;"><b>{n.get('titulo','')}</b></div>
            <div style="margin-top:8px;">{badge}</div>
            <div style="margin-top:10px;color:#0f172a;font-size:14px;line-height:1.35;">{n.get('descripcion','')}</div>
        </div>
        """)

    def _render_task(self, n):
        checklist_items = n.get("checklist") or n.get("acciones", [])
        self._check_widgets = []
        self._input_widgets = []

        action_widgets = [
            widgets.HTML("""
            <div style="margin-top:12px;padding:14px;border-radius:12px;border:1px solid #e2e8f0;background:#ffffff;">
                <div style="font-size:13px;color:#0f172a;"><b>ACCION A EJECUTAR (texto PRO131)</b></div>
            </div>
            """)
        ]

        if checklist_items:
            for item in checklist_items:
                cb = widgets.Checkbox(
                    value=False,
                    description=item,
                    indent=False,
                    layout=widgets.Layout(width="100%")
                )
                self._check_widgets.append(cb)
                action_widgets.append(cb)
        else:
            action_widgets.append(widgets.HTML("<div style='margin-top:8px;font-size:14px;color:#0f172a;'>Sin acciones definidas para este paso.</div>"))

        inputs = n.get("inputs", [])
        if inputs:
            input_box = [widgets.HTML("""
            <div style="margin-top:12px;padding:14px;border-radius:12px;border:1px solid #e2e8f0;background:#ffffff;">
                <div style="font-size:13px;color:#0f172a;"><b>DATOS A REGISTRAR</b></div>
            </div>
            """)]
            for spec in inputs:
                WidgetClass = widgets.Textarea if spec.get("multiline") else widgets.Text
                w = WidgetClass(
                    value="",
                    placeholder=spec.get("placeholder",""),
                    description=spec.get("label",""),
                    style={"description_width":"initial"},
                    layout=widgets.Layout(width="100%")
                )
                w._meta = spec
                self._input_widgets.append(w)
                input_box.append(w)
            action_widgets.append(widgets.VBox(input_box))

        valid = n.get("validacion","")
        action_widgets.append(
            widgets.HTML(f"""
            <div style="margin-top:12px;padding:14px;border-radius:12px;border:2px solid #0ea5e9;background:#ffffff;">
                <div style="font-size:13px;color:#0f172a;"><b>VALIDACION</b></div>
                <div style="margin-top:8px;font-size:16px;color:#0f172a;"><b>{valid}</b></div>
                <div style="margin-top:6px;font-size:12px;color:#0f172a;">Confirma con <b>SI</b> para avanzar. Si respondes <b>NO</b>, el paso queda bloqueado.</div>
            </div>
            """)
        )
        return widgets.VBox(action_widgets)

    def _render_decision(self, n):
        opts = n.get("opciones",[])
        radios = widgets.RadioButtons(
            options=[(o["label"], o["next"]) for o in opts],
            layout={"width":"100%"},
            style={"description_width":"initial"},
        )
        help_txt = n.get("ayuda","")
        help_html = f"<div style='margin-top:10px;font-size:12px;color:#0f172a;opacity:0.9;'><b>Nota:</b> {help_txt}</div>" if help_txt else ""
        return widgets.VBox([
            widgets.HTML(f"""
            <div style="margin-top:12px;padding:14px;border-radius:12px;border:1px solid #e2e8f0;background:#ffffff;">
                <div style="font-size:13px;color:#0f172a;"><b>DECISION (rombo)</b></div>
                <div style="margin-top:8px;font-size:16px;color:#0f172a;"><b>{n.get('pregunta','')}</b></div>
                {help_html}
            </div>
            """),
            radios
        ]), radios

    def _render_footer(self):
        self.btn_volver.disabled = (len(self.historial) == 0)
        return widgets.VBox([
            widgets.HBox([self.btn_si, self.btn_no], layout={"justify_content":"space-between","margin":"10px 0"}),
            self.btn_volver,
            widgets.HTML("<div style='height:10px;'></div>"),
            self.btn_exportar,
            widgets.HTML("<div style='height:10px;'></div>"),
            self.block_panel,
            self.msg_box,
        ])

    def _render(self):
        with self.output:
            self.output.clear_output()
            n = NODOS[self.nodo_id]
            header = self._render_header(n)

            if n["type"] == "task":
                body = self._render_task(n)
                self._decision_widget = None
            elif n["type"] == "decision":
                body, radios = self._render_decision(n)
                self._decision_widget = radios
            elif n["type"] == "end":
                self._decision_widget = None
                body = widgets.HTML(f"""
                <div style="margin-top:12px;padding:18px;border-radius:12px;border:2px solid #22c55e;background:#f0fdf4;">
                    <div style="font-size:20px;color:#0f172a;"><b>FIN / DERIVACION</b></div>
                    <div style="margin-top:10px;font-size:15px;color:#0f172a;">{n.get('mensaje','')}</div>
                    <div style="margin-top:10px;font-size:12px;color:#0f172a;">Estado final: <b>{n.get('estado_final','')}</b></div>
                </div>
                """)
            else:
                body = widgets.HTML("<div>Tipo de nodo no soportado.</div>")

            footer = self._render_footer()
            self.main_box.children = [header, body, footer]
            display(self.main_box)

    def _advance_to(self, next_id):
        prev = self.nodo_id
        self._push_hist(prev)
        self.nodo_id = next_id
        self._log("AVANZA", {"from": prev, "to": next_id})
        self._clear_msg()
        self._render()

    def _on_si(self, _):
        if getattr(self, "is_blocked", False):
            self._set_msg("""<div style='margin-top:10px;padding:12px;border-radius:10px;background:#fffbeb;border:1px solid #f59e0b;color:#0f172a;'>
                <b>Paso bloqueado:</b> Primero registra el motivo y usa <b>Rehacer paso</b> para volver a ejecutar/corregir. Luego valida con <b>SI</b>.
            </div>""")
            return
        n = NODOS[self.nodo_id]
        if n["type"] == "end":
            self._set_msg("<div style='padding:12px;border-radius:10px;background:#e2e8f0;border:1px solid #cbd5e1;color:#0f172a;'><b>Info:</b> Ya estas en un fin/derivacion.</div>")
            return

        if n["type"] == "task":
            faltantes = []
            for cb in getattr(self, "_check_widgets", []):
                desc = getattr(cb, "description", "") or ""
                if "si aplica" in desc.lower():
                    continue
                if not getattr(cb, "value", False):
                    faltantes.append(desc)

            inputs_data = {}
            for w in getattr(self, "_input_widgets", []):
                meta = getattr(w, "_meta", {})
                val = getattr(w, "value", "")
                if meta.get("required") and not str(val).strip():
                    self._set_msg(f"<div style='padding:12px;border-radius:10px;background:#fffbeb;border:1px solid #f59e0b;color:#0f172a;'><b>Falta completar:</b> {meta.get('label','Este campo es obligatorio')}.</div>")
                    return
                inputs_data[meta.get("key")] = val

            if faltantes:
                self._set_msg("<div style='padding:12px;border-radius:10px;background:#fffbeb;border:1px solid #f59e0b;color:#0f172a;'><b>Checklist incompleto:</b> Debes marcar todas las acciones obligatorias antes de avanzar.</div>")
                return

            if inputs_data:
                self._log("INPUTS", inputs_data)

            next_id = n.get("next")
            if next_id:
                self._advance_to(next_id)
            else:
                self._set_msg("<div style='padding:12px;border-radius:10px;background:#fff7ed;border:1px solid #fdba74;color:#0f172a;'><b>Atencion:</b> Este paso no tiene siguiente definido.</div>")
            return

        if n["type"] == "decision":
            if self._decision_widget is None or self._decision_widget.value is None:
                self._set_msg("<div style='padding:12px;border-radius:10px;background:#fffbeb;border:1px solid #f59e0b;color:#0f172a;'><b>Falta seleccion:</b> Elige una opcion para avanzar.</div>")
                return
            self._advance_to(self._decision_widget.value)
            return

    def _motivos_bloqueo(self):
        nid = self.nodo_id
        if nid in ("D0_proveedor_notifico_y_respaldos",):
            return [
                "Proveedor no notificó término del servicio",
                "Proveedor no entregó respaldos mínimos para revisión",
                "Respaldos incompletos/ilegibles – solicitar reenvío",
            ]
        if nid in ("T1_verificar_servicio_vs_oc",):
            return [
                "Servicio no cumple alcance/plazo/calidad pactado en OC",
                "Respaldos técnicos/operacionales insuficientes para V°B°",
            ]
        if nid in ("T2_verificacion_normativa_RE17","T2b_verificacion_normativa_RE16"):
            return [
                "No cumple verificación normativa aplicable (RE.17/RE.16)",
                "Documentación/certificados vencidos o inconsistentes",
            ]
        if nid in ("T3_generar_HES_ML81N","T4_evaluar_proveedor_en_SAP","T5_grabar_HES"):
            return [
                "Datos HES incompletos o inconsistentes (monto/avance/adjuntos)",
                "Faltan respaldos requeridos adjuntos en SAP",
                "Evaluación de proveedor pendiente (obligatoria)",
            ]
        if nid in ("T6_liberar_HES_DOA",):
            return [
                "Falta liberación DOA (responsable objeto de costo)",
                "Liberador no disponible / pendiente de aprobación",
            ]
        if nid in ("T7_comunicar_evaluacion_al_proveedor","T8_notificar_HES_liberada","T9_monitorear_factura"):
            return [
                "Pendiente envío/notificación al proveedor",
                "No hay evidencia de comunicación",
                "Factura no disponible en SAP / inconsistencia con HES/OC",
            ]
        return [
            "Pendiente validación/confirmación del responsable del paso",
            "Se requiere información adicional antes de continuar",
        ]

    def _on_no(self, _):
        n = NODOS[self.nodo_id]
        self.is_blocked = True
        self.block_reason = None
        self.btn_si.disabled = True
        self._log("BLOQUEO", {"titulo": n.get("titulo","")})

        motivos = self._motivos_bloqueo()
        radio = widgets.RadioButtons(
            options=motivos,
            layout={"width":"100%"},
            style={"description_width":"initial"},
        )

        def _on_pick(change):
            if change.get("name") == "value":
                self.block_reason = change["new"]
                self._log("MOTIVO_BLOQUEO_SELECCIONADO", {"motivo": self.block_reason})
                self._set_msg("""
                <div style='margin-top:10px;padding:10px;border-radius:10px;background:#e0f2fe;border:1px solid #0284c7;color:#0f172a;'>
                    <b> Motivo registrado.</b> Ahora ejecuta la corrección indicada y presiona <b>Rehacer paso</b>.
                </div>
                """)

        radio.observe(_on_pick, names="value")

        self.block_panel.children = [
            widgets.HTML("""
            <div style='margin-top:10px;padding:12px;border-radius:12px;background:#fee2e2;border:1px solid #ef4444;color:#0f172a;'>
                <b>🛑 BLOQUEADO:</b> Debes indicar el motivo (quedará en la trazabilidad) y luego rehacer el paso.
            </div>
            """),
            widgets.HTML("<div style='margin-top:6px;font-size:13px;color:#0f172a;'><b>Motivo del bloqueo (según PRO131):</b></div>"),
            radio,
            widgets.HTML("<div style='height:8px;'></div>"),
            self.btn_rehacer,
        ]

        self._set_msg("""
        <div style='margin-top:10px;padding:12px;border-radius:10px;background:#fffbeb;border:1px solid #f59e0b;color:#0f172a;'>
            <b>Instrucción:</b> Selecciona un motivo, realiza la corrección/gestión y luego presiona <b>Rehacer paso</b>.
        </div>
        """)

    def _on_rehacer(self, _):
        if not getattr(self, "is_blocked", False):
            self._set_msg("<div style='margin-top:10px;padding:12px;border-radius:10px;background:#f1f5f9;border:1px solid #cbd5e1;color:#0f172a;'><b>Info:</b> Este paso no está bloqueado.</div>")
            return
        if not self.block_reason:
            self._set_msg("<div style='margin-top:10px;padding:12px;border-radius:10px;background:#fffbeb;border:1px solid #f59e0b;color:#0f172a;'><b>Falta motivo:</b> Debes seleccionar un motivo de bloqueo antes de rehacer.</div>")
            return

        self._log("REHACER_PASO", {"motivo": self.block_reason})
        self.is_blocked = False
        self.btn_si.disabled = False
        self.block_panel.children = []
        self._set_msg("<div style='margin-top:10px;padding:12px;border-radius:10px;background:#e8f5e9;border:1px solid #22c55e;color:#0f172a;'><b> Paso listo para rehacer.</b> Ejecuta la acción del paso actual y valida con <b>SÍ</b>.</div>")

    def _on_volver(self, _):
        prev = self._pop_hist()
        self.is_blocked = False
        self.block_reason = None
        self.block_panel.children = []
        self.btn_si.disabled = False
        if prev is None:
            self._set_msg("<div style='padding:12px;border-radius:10px;background:#e2e8f0;border:1px solid #cbd5e1;color:#0f172a;'><b>Info:</b> Ya estás en el inicio.</div>")
            return
        cur = self.nodo_id
        self.nodo_id = prev
        self._log("VOLVER", {"from": cur, "to": prev})
        self._clear_msg()
        self._render()

    def _on_exportar(self, _):
        payload = {
            "proceso": "PRO131 – Recepción de Servicios (v7 nov-2024)",
            "session_id": str(uuid.uuid4()),
            "export_ts": _now_iso(),
            "current_node": self.nodo_id,
            "history_stack": list(self.historial),
            "logs": list(self.logs),
        }
        pretty = json.dumps(payload, ensure_ascii=False, indent=2)
        self._set_msg(f"""
        <div style='margin-top:10px;padding:12px;border-radius:10px;background:#f1f5f9;border:1px solid #cbd5e1;color:#0f172a;'>
            <b> Export JSON (trazabilidad)</b>
            <pre style='white-space:pre-wrap;margin-top:10px;color:#0f172a;'>{pretty}</pre>
        </div>
        """)

    def iniciar(self):
        display(self.output)

hmi = PRO131HMI()
hmi.iniciar()
